# Module 10: Propensity Scores and the Overlap Assumption

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

A propensity score is the probability of treatment given covariates.
Matching, weighting and stratifying on it all rest on **overlap**: for every
covariate value, both treated and untreated units must be possible.

This module fits one, finds it separates perfectly, and then shows that the
separation has **two independent causes** and only one of them is about this
program.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

In [ ]:
X = profile.set_index("agency_id").copy()
X = X.loc[sorted(X.index)]
X["treated"] = [1 if a in TRAINED else 0 for a in X.index]
X["lpop"] = np.log(X["population_served"])
X["lsworn"] = np.log(X["sworn_officers"])
print(f"{len(X)} agencies, {X['treated'].sum()} of them treated")

## 2. Separation arrives as covariates are added

In [ ]:
import warnings as _w
_w.filterwarnings("ignore")

sets = [("one covariate", ["lsworn"]),
        ("two covariates", ["lsworn", "violent_crime_rate_per_1000"]),
        ("four covariates", ["lsworn", "lpop", "violent_crime_rate_per_1000",
                             "property_crime_rate_per_1000"])]
for lab, cols in sets:
    Z = sm.add_constant(X[cols].astype(float))
    lg = sm.Logit(X["treated"], Z).fit(disp=False)
    ps = lg.predict(Z)
    print(f"  {lab:18s} pseudo R2 {lg.prsquared:.3f}   "
          f"lowest treated score {ps[X['treated'] == 1].min():.3f}   "
          f"highest control score {ps[X['treated'] == 0].max():.3f}")

With four covariates the two groups are perfectly separated: no treated agency
scores below any control agency. **None of those four is the variable
selection was based on.**

## 3. How much of that is mechanical

With twelve units, four covariates can separate five from seven by accident.
Test it with columns that contain nothing.

In [ ]:
import warnings as _w
_w.filterwarnings("ignore")          # perfect separation warns; that is the point

rng = np.random.default_rng(5)
seps = 0
for _ in range(200):
    Y = X.copy()
    for j in range(4):
        Y[f"r{j}"] = rng.normal(size=len(Y))
    Z = sm.add_constant(Y[[f"r{j}" for j in range(4)]])
    try:
        lg = sm.Logit(Y["treated"], Z).fit(disp=False)
        seps += lg.prsquared > 0.99
    except Exception:
        seps += 1
print(f"  four columns of pure noise separate these groups perfectly "
      f"in {seps} of 200 draws ({100 * seps / 200:.0f} percent)")

Six percent of the time, noise alone does it. **Perfect separation at twelve
units is partly a small sample artefact and should not be read as evidence
about the assignment mechanism.**

That matters for the usual response, which is to drop covariates until the
logit converges. Doing so produces a propensity score that looks usable and
rests on a model chosen for its convergence rather than for its content.

## 4. The overlap that actually matters

Separation in a fitted score is a symptom. The question is whether the
**variable selection was based on** overlaps, and that can be looked at
directly.

In [ ]:
tv = sorted(X[X["treated"] == 1]["pre_program_uof_per_100_arrests"])
cv = sorted(X[X["treated"] == 0]["pre_program_uof_per_100_arrests"])
print(f"  treated baseline rates: {[round(v, 3) for v in tv]}")
print(f"  control baseline rates: {[round(v, 3) for v in cv]}")
print(f"\n  the lowest treated agency is at {min(tv):.3f}")
print(f"  {sum(v > min(tv) for v in cv)} control agency lies above it")
for c, lab in [("sworn_officers", "sworn officers"),
               ("violent_crime_rate_per_1000", "violent crime rate"),
               ("pre_program_uof_per_100_arrests", "USE OF FORCE RATE BEFORE")]:
    t, ct = X[X["treated"] == 1][c], X[X["treated"] == 0][c]
    ov = max(0, min(t.max(), ct.max()) - max(t.min(), ct.min()))
    rng_ = max(t.max(), ct.max()) - min(t.min(), ct.min())
    print(f"  {lab:26s} the groups overlap over {100 * ov / rng_:.0f} percent "
          f"of the combined range")

The covariates overlap substantially. **The outcome's own history overlaps
over four percent of its range**, and the entire overlap is one agency,
Havenbrook at 3.200, sitting just above Summit County at 3.118.

So there is exactly one control agency available to match four of the five
treated ones, and it cannot be used four times.

**This is not a failure of the estimator. It is the selection rule.** Four of
the top five agencies by baseline rate were taken, so by construction there is
almost nothing at the top of the distribution left untreated.

## 5. What can be done

| Response | Verdict here |
|---|---|
| Trim to the region of common support | leaves one control agency; not usable |
| Drop the outcome history from the score | estimates a score that omits the selection rule |
| Use a coarser score, fewer covariates | fits, and does not address the overlap |
| Report the lack of overlap and change estimand | **this is the answer** |
| Difference in differences instead | does not require overlap in levels, only parallel trends |

**Difference in differences is the right tool precisely because it does not
need overlap in the level.** It needs the groups to move together, which is a
different and weaker requirement, and it is the one Intermediate
[Module 7](../../Intermediate/Notebooks/Module_07_Testing_Parallel_Trends.ipynb)
tests.

## Exercise

Trim the sample to the region of common support and see what is left.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    lo_t = X[X["treated"] == 1]["pre_program_uof_per_100_arrests"].min()
    hi_c = X[X["treated"] == 0]["pre_program_uof_per_100_arrests"].max()
    keep_ids = X[(X["pre_program_uof_per_100_arrests"] >= lo_t)
                 & (X["pre_program_uof_per_100_arrests"] <= hi_c)].index
    print(f"  common support runs from {lo_t:.3f} to {hi_c:.3f}\n")
    for a in keep_ids:
        print(f"    {NAME[a]:34s} "
              f"{X.loc[a, 'pre_program_uof_per_100_arrests']:.3f}  "
              f"{'treated' if a in TRAINED else 'control'}")
    print(f"\n  {len(keep_ids)} agencies survive trimming, "
          f"{sum(a in TRAINED for a in keep_ids)} of them treated")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Two agencies survive: Summit County, treated, and Havenbrook, control.

**And Summit County is the agency excluded from the main analysis** for a pre
program trend of 12 percent a year. The trimmed sample is one treated unit,
one control, and the treated unit is the one known to violate the design's
assumption.

That is the honest answer to "why not use matching here", and it takes four
lines to produce. **Trimming to common support is the diagnostic, not just the
remedy**: when it leaves two units, the overlap assumption has failed and no
weighting scheme repairs it.

The finding is also a caution about reporting balance. A propensity score
fitted on the four covariates in section 2 would show excellent balance after
weighting, because those covariates do overlap. The balance table would be
reassuring and the estimate would rest on nothing.

</details>

---

**Next:** [Module 11: Instrumental Variables](Module_11_Instrumental_Variables.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*